# Multimodal Deepfake Detection System - Colab / Kaggle GPU Training

This notebook trains the **Multimodal Cross-Attention Deepfake Detector** on a free **Colab / Kaggle GPU**.

### Ground Rules Compliance
- 100% written from scratch: no deepfake pretrained weights.
- Standard libraries: PyTorch, torchvision, OpenCV, scikit-learn.
- Leak-free subject-level splitting (70/15/15).

## 1. Verify GPU Acceleration

In [ ]:
!nvidia-smi
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active GPU: {torch.cuda.get_device_name(0)}")

## 2. Install Required Dependencies

In [ ]:
%pip install -q pyyaml scikit-learn opencv-python pillow matplotlib tqdm
print("Dependencies installed successfully.")

## 3. Generate or Mount Dataset & Index Labels
Mount your Google Drive or generate starter synthetic portraits to verify the entire pipeline.

In [ ]:
# Optional: from google.colab import drive; drive.mount('/content/drive')

# Generate starter balanced dataset for quick pipeline verification
!python tests/generate_test_samples.py

# Index directory hierarchy into leak-free subject labels
!python build_labels.py --data_dir data/raw --output_csv data/labels.csv

## 4. Pre-Cache Features (15x Speedup)
Pre-computes face crops (112x112), 2D Fourier (FFT) spectra, and 32-dim handcrafted features.

In [ ]:
!python -m src.features.cache_features --labels_csv data/labels.csv --output_dir data/cache

## 5. Train Multimodal Cross-Attention Detector on GPU

In [ ]:
device_flag = "cuda" if torch.cuda.is_available() else "cpu"
!python -m src.training.train --config configs/default.yaml --epochs 25 --device {device_flag}

## 6. Run Scientific Ablation & Robustness Stress Testing

In [ ]:
!python -m src.training.evaluate --config configs/default.yaml

## 7. Visualize Results & Download Artifacts

In [ ]:
import json
from IPython.display import Image, display

# Display generated ROC curve and confusion matrix if available
if os.path.exists("results/roc_curve.png"):
    display(Image("results/roc_curve.png"))
if os.path.exists("results/confusion_matrix.png"):
    display(Image("results/confusion_matrix.png"))

with open("results/ablation_study.json") as f:
    print(json.dumps(json.load(f), indent=2))